# Матрица компетенций и форм контроля

Эта вычислительная тетрадь показывает, как компетенции ПК-ИИ-1 — ПК-ИИ-10 связаны с модулями курса и формами контроля. Все данные загружаются из локального файла `data/structured_data.yaml`.

## 1. Загрузка данных

Данные хранятся отдельно от текста главы, чтобы их можно было использовать повторно в инженерной среде.

In [ ]:
from pathlib import Path
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

base = Path.cwd()
if not (base / 'data' / 'structured_data.yaml').exists():
    candidates = list(Path.cwd().glob('**/branch_05_fos/data/structured_data.yaml'))
    if candidates:
        base = candidates[0].parents[1]

with open(base / 'data' / 'structured_data.yaml', encoding='utf-8') as f:
    data = yaml.safe_load(f)

(base / 'assets' / 'diagrams').mkdir(parents=True, exist_ok=True)
print(data['branch']['title'])

## 2. Таблица компетенций

Таблица показывает формулировки компетенций, артефакты и проверочные элементы.

In [ ]:
competencies = pd.DataFrame(data['competencies'])
competencies[['code', 'name', 'artifact', 'test']]

## 3. Матрица «модуль — компетенция»

Матрица нужна для проверки полноты фонда оценочных средств. Если компетенция не связана ни с одним модулем или не имеет проверки, это методический дефект.

In [ ]:
modules = pd.DataFrame(data['modules'])
competency_codes = [c['code'] for c in data['competencies']]
module_ids = [m['id'] for m in data['modules']]
module_titles = {m['id']: f"{m['number']}. {m['title']}" for m in data['modules']}

matrix = pd.DataFrame(0, index=[module_titles[m] for m in module_ids], columns=competency_codes)
for comp in data['competencies']:
    for module_id in comp['modules']:
        matrix.loc[module_titles[module_id], comp['code']] = 1

matrix

## 4. Визуализация матрицы

Значение 1 означает, что компетенция формируется или подтверждается в модуле.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.imshow(matrix.values, aspect='auto')
ax.set_xticks(range(len(matrix.columns)))
ax.set_xticklabels(matrix.columns, rotation=45, ha='right')
ax.set_yticks(range(len(matrix.index)))
ax.set_yticklabels(matrix.index)
ax.set_title('Матрица соответствия модулей и компетенций')
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        ax.text(j, i, str(matrix.values[i, j]), ha='center', va='center')
plt.tight_layout()
path = base / 'assets' / 'diagrams' / 'module_competency_matrix.png'
plt.savefig(path, dpi=200, bbox_inches='tight')
plt.show()
print(path)

## 5. Граф связей модулей, компетенций и контрольных форм

Граф показывает, какие контрольные элементы подтверждают компетенции внутри модулей.

In [ ]:
G = nx.Graph()
for module in data['modules']:
    module_label = f"М{module['number']}"
    G.add_node(module_label, kind='module')
for comp in data['competencies']:
    G.add_node(comp['code'], kind='competency')
    for module_id in comp['modules']:
        module = next(m for m in data['modules'] if m['id'] == module_id)
        G.add_edge(f"М{module['number']}", comp['code'])
for control in data['module_controls']:
    control_label = f"Контроль {control['module'].upper()}"
    G.add_node(control_label, kind='control')
    module = next(m for m in data['modules'] if m['id'] == control['module'])
    G.add_edge(f"М{module['number']}", control_label)
    for comp in control['competencies']:
        G.add_edge(control_label, comp)

fig, ax = plt.subplots(figsize=(14, 9))
pos = nx.spring_layout(G, seed=7, k=0.9)
nx.draw_networkx(G, pos=pos, ax=ax, with_labels=True, font_size=8, node_size=1200)
ax.set_title('Граф связей: модули, компетенции и контроль')
ax.axis('off')
plt.tight_layout()
path = base / 'assets' / 'diagrams' / 'competency_control_graph.png'
plt.savefig(path, dpi=200, bbox_inches='tight')
plt.show()
print(path)

## 6. Вывод

Фонд оценочных средств покрывает все 10 компетенций. Каждая компетенция связана с артефактом и проверкой. Полученные схемы можно использовать в главе Jupyter Book и в техническом задании для реализации контрольных элементов в платформе.